In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes

from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = "enares-2024-crs04"
LOCATION = "US"
EXPECTED_ROWS = 18807

ROOT_DRIVE = Path(
    "/content/drive/MyDrive/ENARES_2024_PROJECT"
)

LOG_DIR = ROOT_DRIVE / "05Resultados" / "logs" / "stage03"
SQL_DIR = ROOT_DRIVE / "02SQL"
DOCS_DIR = ROOT_DRIVE / "docs"

for directory in [LOG_DIR, SQL_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

A = (
    f"{PROJECT_ID}."
    "enares2024_crs04_analytical."
    "analytical_crs04_adolescents"
)

print("Tabla analítica:", A)

Mounted at /content/drive
Tabla analítica: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents


In [2]:
try:
    analytical_table = client.get_table(A)
except Exception as error:
    raise RuntimeError(
        "No existe analytical_crs04_adolescents. "
        "Ejecuta primero NB02."
    ) from error

analytical_status = pd.DataFrame([
    {
        "table": A,
        "rows": analytical_table.num_rows,
        "columns": len(analytical_table.schema),
        "verified_utc": RUN_UTC,
    }
])

display(analytical_status)

analytical_status.to_csv(
    LOG_DIR / "stage3_nb03_analytical_prerequisite.csv",
    index=False
)

if analytical_table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(
        f"Analytical tiene {analytical_table.num_rows} filas; "
        f"se esperaban {EXPECTED_ROWS}."
    )

required_previous_indicators = [
    "VP_HOGAR",
    "VF_HOGAR",
    "VF_HOGAR_01",
    "VF_HOGAR_03",
    "VN_HOGAR1",
    "INDICADOR_8_3_6",
    "VP_o_VF_HOGAR",
    "VP_VF_HOGAR",
]

existing_columns = {
    field.name
    for field in analytical_table.schema
}

missing_previous = sorted(
    set(required_previous_indicators) - existing_columns
)

if missing_previous:
    raise RuntimeError(
        "NB02 no está completo. Faltan indicadores: "
        + ", ".join(missing_previous)
    )

print(
    "Prerequisito aprobado: NB02 existe, conserva 18,807 filas "
    "y contiene los indicadores 3.1–3.2."
)

,table,rows,columns,verified_utc
0,enares-2024-crs04.enares2024_crs04_analytical....,18807,1295,2026-07-17T02:00:03.319372+00:00


Prerequisito aprobado: NB02 existe, conserva 18,807 filas y contiene los indicadores 3.1–3.2.


In [3]:
# ============================================================
# Traducción SPSS 3.3 — VP_ESCUELA
# Fuente:
# 09_CRS04_3.3 Violencia en el entorno escolar_ver4.sps
#
# Este notebook NO reconstruye analytical desde cleaned.
# Toma la tabla producida por NB02 y agrega VP_ESCUELA.
# ============================================================

required_vp_escuela = ["C3P225"]

for i in range(1, 15):
    required_vp_escuela.extend([
        f"C3P223_{i}",
        f"C3P223A_{i}",
        f"C3P223C_{i}",
        f"C3P223E_{i}",
    ])

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_vp_escuela = sorted(
    set(required_vp_escuela) - existing_columns
)

if missing_vp_escuela:
    raise RuntimeError(
        "No se puede crear VP_ESCUELA. "
        "Faltan variables fuente: "
        + ", ".join(missing_vp_escuela)
    )

school_item_conditions = []

for i in range(1, 15):
    school_item_conditions.append(f"""
    (
      `C3P223_{i}` = 1
      AND (
        `C3P223A_{i}` = 1
        OR `C3P223C_{i}` = 1
        OR `C3P223E_{i}` = 1
      )
    )
    """)

any_school_item = "\nOR\n".join(
    school_item_conditions
)

select_prefix = (
    "* EXCEPT(VP_ESCUELA)"
    if "VP_ESCUELA" in existing_columns
    else "*"
)

vp_escuela_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN C3P225 = 1
     AND (
       {any_school_item}
     )
    THEN 1
    ELSE 0
  END AS VP_ESCUELA

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_33_vp_escuela.sql").write_text(
    vp_escuela_sql,
    encoding="utf-8"
)

print(vp_escuela_sql)

client.query(vp_escuela_sql).result()

print("VP_ESCUELA agregado a la tabla analytical existente.")


CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN C3P225 = 1
     AND (
       
    (
      `C3P223_1` = 1
      AND (
        `C3P223A_1` = 1
        OR `C3P223C_1` = 1
        OR `C3P223E_1` = 1
      )
    )
    
OR

    (
      `C3P223_2` = 1
      AND (
        `C3P223A_2` = 1
        OR `C3P223C_2` = 1
        OR `C3P223E_2` = 1
      )
    )
    
OR

    (
      `C3P223_3` = 1
      AND (
        `C3P223A_3` = 1
        OR `C3P223C_3` = 1
        OR `C3P223E_3` = 1
      )
    )
    
OR

    (
      `C3P223_4` = 1
      AND (
        `C3P223A_4` = 1
        OR `C3P223C_4` = 1
        OR `C3P223E_4` = 1
      )
    )
    
OR

    (
      `C3P223_5` = 1
      AND (
        `C3P223A_5` = 1
        OR `C3P223C_5` = 1
        OR `C3P223E_5` = 1
      )
    )
    
OR

    (
      `C3P223_6` = 1
      AND (
        `C3P223A_6` = 1
        OR `C3P223C_6` = 1
        OR `C3P223E_6` = 1
      )
    )
    

In [4]:
vp_escuela_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VP_ESCUELA IS NULL
    OR VP_ESCUELA NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VP_ESCUELA = 0) AS zero_values,
  COUNTIF(VP_ESCUELA = 1) AS one_values

FROM `{A}`
""").result().to_dataframe()

vp_escuela_validation.to_csv(
    LOG_DIR / "stage3_vp_escuela_validation.csv",
    index=False
)

display(vp_escuela_validation)

r = vp_escuela_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VP_ESCUELA alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VP_ESCUELA contiene valores fuera de 0/1."
    )

print(
    "VP_ESCUELA validado: 18,807 filas y dominio 0/1."
)

,total_rows,invalid_values,zero_values,one_values
0,18807,0,11285,7522


VP_ESCUELA validado: 18,807 filas y dominio 0/1.


In [5]:
# ============================================================
# Traducción SPSS 3.3 — VF_ESCUELA
# Fuente:
# 09_CRS04_3.3 Violencia en el entorno escolar_ver4.sps
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

# 1. Verificar variables fuente
required_vf_escuela = ["C3P229"]

for i in range(1, 11):
    required_vf_escuela.extend([
        f"C3P227_{i}",
        f"C3P227A_{i}",
        f"C3P227C_{i}",
        f"C3P227E_{i}",
    ])

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_vf_escuela = sorted(
    set(required_vf_escuela) - existing_columns
)

if missing_vf_escuela:
    raise RuntimeError(
        "No se puede crear VF_ESCUELA. "
        "Faltan variables fuente: "
        + ", ".join(missing_vf_escuela)
    )

print("Variables fuente de VF_ESCUELA verificadas.")


# 2. Construir condición de los 10 ítems
vf_school_conditions = []

for i in range(1, 11):
    vf_school_conditions.append(f"""
    (
      `C3P227_{i}` = 1
      AND (
        `C3P227A_{i}` = 1
        OR `C3P227C_{i}` = 1
        OR `C3P227E_{i}` = 1
      )
    )
    """)

any_vf_school_item = "\nOR\n".join(vf_school_conditions)


# 3. Preparar reejecución segura
select_prefix = (
    "* EXCEPT(VF_ESCUELA)"
    if "VF_ESCUELA" in existing_columns
    else "*"
)


# 4. Crear o reemplazar VF_ESCUELA
vf_escuela_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN C3P229 = 1
     AND (
       {any_vf_school_item}
     )
    THEN 1
    ELSE 0
  END AS VF_ESCUELA

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_33_vf_escuela.sql").write_text(
    vf_escuela_sql,
    encoding="utf-8"
)

print(vf_escuela_sql)

client.query(vf_escuela_sql).result()

print("VF_ESCUELA agregado correctamente.")


# 5. Validar
vf_escuela_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VF_ESCUELA IS NULL
    OR VF_ESCUELA NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VF_ESCUELA = 0) AS zero_values,
  COUNTIF(VF_ESCUELA = 1) AS one_values

FROM `{A}`
""").result().to_dataframe()

vf_escuela_validation.to_csv(
    LOG_DIR / "stage3_vf_escuela_validation.csv",
    index=False
)

display(vf_escuela_validation)

r = vf_escuela_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VF_ESCUELA alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VF_ESCUELA contiene valores fuera de 0/1."
    )

print("VF_ESCUELA validado: 18,807 filas y dominio 0/1.")

Variables fuente de VF_ESCUELA verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN C3P229 = 1
     AND (
       
    (
      `C3P227_1` = 1
      AND (
        `C3P227A_1` = 1
        OR `C3P227C_1` = 1
        OR `C3P227E_1` = 1
      )
    )
    
OR

    (
      `C3P227_2` = 1
      AND (
        `C3P227A_2` = 1
        OR `C3P227C_2` = 1
        OR `C3P227E_2` = 1
      )
    )
    
OR

    (
      `C3P227_3` = 1
      AND (
        `C3P227A_3` = 1
        OR `C3P227C_3` = 1
        OR `C3P227E_3` = 1
      )
    )
    
OR

    (
      `C3P227_4` = 1
      AND (
        `C3P227A_4` = 1
        OR `C3P227C_4` = 1
        OR `C3P227E_4` = 1
      )
    )
    
OR

    (
      `C3P227_5` = 1
      AND (
        `C3P227A_5` = 1
        OR `C3P227C_5` = 1
        OR `C3P227E_5` = 1
      )
    )
    
OR

    (
      `C3P227_6` = 1
      AND (
        `C3P227A_6` = 1
        OR `C3P227C_6` = 1
 

,total_rows,invalid_values,zero_values,one_values
0,18807,0,16376,2431


VF_ESCUELA validado: 18,807 filas y dominio 0/1.


In [6]:
# ============================================================
# Traducción SPSS 3.3 — VP_o_VF_ESCUELA
# Violencia psicológica y/o física en la escuela
#
# Regla:
# VP_o_VF_ESCUELA = 1 cuando
#   VP_ESCUELA = 1 OR VF_ESCUELA = 1
# En cualquier otro caso = 0
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

required_vp_o_vf_escuela = [
    "VP_ESCUELA",
    "VF_ESCUELA",
]

missing_vp_o_vf_escuela = sorted(
    set(required_vp_o_vf_escuela) - existing_columns
)

if missing_vp_o_vf_escuela:
    raise RuntimeError(
        "No se puede crear VP_o_VF_ESCUELA. "
        "Faltan variables fuente: "
        + ", ".join(missing_vp_o_vf_escuela)
    )

print("Variables fuente de VP_o_VF_ESCUELA verificadas.")


select_prefix = (
    "* EXCEPT(VP_o_VF_ESCUELA)"
    if "VP_o_VF_ESCUELA" in existing_columns
    else "*"
)


vp_o_vf_escuela_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN VP_ESCUELA = 1
      OR VF_ESCUELA = 1
    THEN 1
    ELSE 0
  END AS VP_o_VF_ESCUELA

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_33_vp_o_vf_escuela.sql").write_text(
    vp_o_vf_escuela_sql,
    encoding="utf-8"
)

print(vp_o_vf_escuela_sql)

client.query(vp_o_vf_escuela_sql).result()

print("VP_o_VF_ESCUELA creado correctamente.")


vp_o_vf_escuela_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VP_o_VF_ESCUELA IS NULL
    OR VP_o_VF_ESCUELA NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VP_o_VF_ESCUELA = 0) AS zero_values,

  COUNTIF(VP_o_VF_ESCUELA = 1) AS one_values,

  COUNTIF(
    (VP_ESCUELA = 1 OR VF_ESCUELA = 1)
    AND VP_o_VF_ESCUELA != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    VP_ESCUELA != 1
    AND VF_ESCUELA != 1
    AND VP_o_VF_ESCUELA != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vp_o_vf_escuela_validation.to_csv(
    LOG_DIR / "stage3_vp_o_vf_escuela_validation.csv",
    index=False
)

display(vp_o_vf_escuela_validation)

r = vp_o_vf_escuela_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VP_o_VF_ESCUELA alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VP_o_VF_ESCUELA contiene valores fuera de 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VP_o_VF_ESCUELA no coincide con la lógica esperada."
    )

print(
    "VP_o_VF_ESCUELA validado: "
    "18,807 filas, dominio 0/1 y consistencia exacta."
)


vp_o_vf_escuela_distribution = client.query(f"""
SELECT
  VP_o_VF_ESCUELA,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VP_o_VF_ESCUELA
ORDER BY VP_o_VF_ESCUELA
""").result().to_dataframe()

vp_o_vf_escuela_distribution.to_csv(
    LOG_DIR / "stage3_vp_o_vf_escuela_distribution.csv",
    index=False
)

display(vp_o_vf_escuela_distribution)

Variables fuente de VP_o_VF_ESCUELA verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN VP_ESCUELA = 1
      OR VF_ESCUELA = 1
    THEN 1
    ELSE 0
  END AS VP_o_VF_ESCUELA

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VP_o_VF_ESCUELA creado correctamente.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,10763,8044,0,0


VP_o_VF_ESCUELA validado: 18,807 filas, dominio 0/1 y consistencia exacta.


,VP_o_VF_ESCUELA,n
0,0,10763
1,1,8044


In [7]:
# ============================================================
# Traducción SPSS 3.3 — VP_VF_ESCUELA
# Coocurrencia de violencia psicológica y física en la escuela
#
# SPSS:
# IF (VP_ESCUELA = 1 AND VF_ESCUELA = 1)
#   VP_VF_ESCUELA = 1.
# RECODE VP_VF_ESCUELA (SYSMIS = 0).
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

required_vp_vf_escuela = [
    "VP_ESCUELA",
    "VF_ESCUELA",
]

missing_vp_vf_escuela = sorted(
    set(required_vp_vf_escuela) - existing_columns
)

if missing_vp_vf_escuela:
    raise RuntimeError(
        "No se puede crear VP_VF_ESCUELA. "
        "Faltan variables fuente: "
        + ", ".join(missing_vp_vf_escuela)
    )

print("Variables fuente de VP_VF_ESCUELA verificadas.")


# Permitir reejecución sin duplicar la columna
select_prefix = (
    "* EXCEPT(VP_VF_ESCUELA)"
    if "VP_VF_ESCUELA" in existing_columns
    else "*"
)


vp_vf_escuela_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN VP_ESCUELA = 1
     AND VF_ESCUELA = 1
    THEN 1

    -- Equivale a RECODE (SYSMIS = 0)
    ELSE 0
  END AS VP_VF_ESCUELA

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_33_vp_vf_escuela.sql").write_text(
    vp_vf_escuela_sql,
    encoding="utf-8"
)

print(vp_vf_escuela_sql)

client.query(vp_vf_escuela_sql).result()

print("VP_VF_ESCUELA creado correctamente.")


# ============================================================
# Validación
# ============================================================

vp_vf_escuela_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VP_VF_ESCUELA IS NULL
    OR VP_VF_ESCUELA NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VP_VF_ESCUELA = 0) AS zero_values,
  COUNTIF(VP_VF_ESCUELA = 1) AS one_values,

  COUNTIF(
    VP_ESCUELA = 1
    AND VF_ESCUELA = 1
    AND VP_VF_ESCUELA != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    NOT (VP_ESCUELA = 1 AND VF_ESCUELA = 1)
    AND VP_VF_ESCUELA != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vp_vf_escuela_validation.to_csv(
    LOG_DIR / "stage3_vp_vf_escuela_validation.csv",
    index=False
)

display(vp_vf_escuela_validation)

r = vp_vf_escuela_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VP_VF_ESCUELA alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VP_VF_ESCUELA contiene valores fuera de 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VP_VF_ESCUELA no coincide con la lógica SPSS."
    )

print(
    "VP_VF_ESCUELA validado: "
    "18,807 filas, dominio 0/1 y consistencia exacta."
)


# Distribución
vp_vf_escuela_distribution = client.query(f"""
SELECT
  VP_VF_ESCUELA,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VP_VF_ESCUELA
ORDER BY VP_VF_ESCUELA
""").result().to_dataframe()

vp_vf_escuela_distribution.to_csv(
    LOG_DIR / "stage3_vp_vf_escuela_distribution.csv",
    index=False
)

display(vp_vf_escuela_distribution)

Variables fuente de VP_VF_ESCUELA verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN VP_ESCUELA = 1
     AND VF_ESCUELA = 1
    THEN 1

    -- Equivale a RECODE (SYSMIS = 0)
    ELSE 0
  END AS VP_VF_ESCUELA

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

VP_VF_ESCUELA creado correctamente.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,16898,1909,0,0


VP_VF_ESCUELA validado: 18,807 filas, dominio 0/1 y consistencia exacta.


,VP_VF_ESCUELA,n
0,0,16898
1,1,1909


In [8]:
# ============================================================
# Traducción SPSS 3.3 — VS_ESCUELA
# Violencia sexual en el entorno escolar, últimos 12 meses
#
# Fuente:
# 09_CRS04_3.3 Violencia en el entorno escolar_ver4.sps
#
# Regla SPSS:
# VS_ESCUELA = 1 si al menos uno de los 16 ítems cumple:
#   C4P248_i = 1
#   AND C4P248C_i = 1
#   AND (C4P248A_27_i = 1 OR C4P248A_28_i = 1)
#
# Los demás casos pasan a 0 mediante RECODE (SYSMIS=0).
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

# ------------------------------------------------------------
# 1. Verificar variables fuente
# ------------------------------------------------------------

required_vs_escuela = []

for i in range(1, 17):
    required_vs_escuela.extend([
        f"C4P248_{i}",
        f"C4P248A_27_{i}",
        f"C4P248A_28_{i}",
        f"C4P248C_{i}",
    ])

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_vs_escuela = sorted(
    set(required_vs_escuela) - existing_columns
)

if missing_vs_escuela:
    raise RuntimeError(
        "No se puede crear VS_ESCUELA. "
        "Faltan variables fuente: "
        + ", ".join(missing_vs_escuela)
    )

print("Variables fuente de VS_ESCUELA verificadas.")


# ------------------------------------------------------------
# 2. Construir condición de los 16 ítems
# ------------------------------------------------------------

vs_school_conditions = []

for i in range(1, 17):
    vs_school_conditions.append(f"""
    (
      `C4P248_{i}` = 1
      AND `C4P248C_{i}` = 1
      AND (
        `C4P248A_27_{i}` = 1
        OR `C4P248A_28_{i}` = 1
      )
    )
    """)

any_vs_school_item = "\nOR\n".join(
    vs_school_conditions
)


# ------------------------------------------------------------
# 3. Preparar reejecución segura
# ------------------------------------------------------------

select_prefix = (
    "* EXCEPT(VS_ESCUELA)"
    if "VS_ESCUELA" in existing_columns
    else "*"
)


# ------------------------------------------------------------
# 4. Crear o reemplazar VS_ESCUELA
# ------------------------------------------------------------

vs_escuela_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN (
      {any_vs_school_item}
    )
    THEN 1

    -- Equivale a RECODE VS_ESCUELA (SYSMIS=0)
    ELSE 0
  END AS VS_ESCUELA

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_33_vs_escuela.sql").write_text(
    vs_escuela_sql,
    encoding="utf-8"
)

print(vs_escuela_sql)

client.query(vs_escuela_sql).result()

print("VS_ESCUELA creado correctamente.")


# ------------------------------------------------------------
# 5. Validar universo, dominio y correspondencia
# ------------------------------------------------------------

vs_escuela_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    VS_ESCUELA IS NULL
    OR VS_ESCUELA NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(VS_ESCUELA = 0) AS zero_values,
  COUNTIF(VS_ESCUELA = 1) AS one_values,

  COUNTIF(
    (
      {any_vs_school_item}
    )
    AND VS_ESCUELA != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    NOT (
      {any_vs_school_item}
    )
    AND VS_ESCUELA != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

vs_escuela_validation.to_csv(
    LOG_DIR / "stage3_vs_escuela_validation.csv",
    index=False
)

display(vs_escuela_validation)

r = vs_escuela_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "VS_ESCUELA alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "VS_ESCUELA contiene valores fuera de 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "VS_ESCUELA no coincide con la lógica SPSS."
    )

print(
    "VS_ESCUELA validado: "
    "18,807 filas, dominio 0/1 y consistencia exacta."
)


# ------------------------------------------------------------
# 6. Distribución
# ------------------------------------------------------------

vs_escuela_distribution = client.query(f"""
SELECT
  VS_ESCUELA,
  COUNT(*) AS n
FROM `{A}`
GROUP BY VS_ESCUELA
ORDER BY VS_ESCUELA
""").result().to_dataframe()

vs_escuela_distribution.to_csv(
    LOG_DIR / "stage3_vs_escuela_distribution.csv",
    index=False
)

display(vs_escuela_distribution)

Variables fuente de VS_ESCUELA verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN (
      
    (
      `C4P248_1` = 1
      AND `C4P248C_1` = 1
      AND (
        `C4P248A_27_1` = 1
        OR `C4P248A_28_1` = 1
      )
    )
    
OR

    (
      `C4P248_2` = 1
      AND `C4P248C_2` = 1
      AND (
        `C4P248A_27_2` = 1
        OR `C4P248A_28_2` = 1
      )
    )
    
OR

    (
      `C4P248_3` = 1
      AND `C4P248C_3` = 1
      AND (
        `C4P248A_27_3` = 1
        OR `C4P248A_28_3` = 1
      )
    )
    
OR

    (
      `C4P248_4` = 1
      AND `C4P248C_4` = 1
      AND (
        `C4P248A_27_4` = 1
        OR `C4P248A_28_4` = 1
      )
    )
    
OR

    (
      `C4P248_5` = 1
      AND `C4P248C_5` = 1
      AND (
        `C4P248A_27_5` = 1
        OR `C4P248A_28_5` = 1
      )
    )
    
OR

    (
      `C4P248_6` = 1
      AND `C4P248C_6` = 1
      AND (
        `C4P248A_27_6`

,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,16657,2150,0,0


VS_ESCUELA validado: 18,807 filas, dominio 0/1 y consistencia exacta.


,VS_ESCUELA,n
0,0,16657
1,1,2150


In [9]:
# ============================================================
# Traducción SPSS 3.3 — INDICADOR_8_3_9
# Violencia en el entorno escolar, indicador consolidado
#
# Regla:
# INDICADOR_8_3_9 = 1 cuando:
#   VP_ESCUELA = 1
#   OR VF_ESCUELA = 1
#   OR VS_ESCUELA = 1
#
# Compatible con BigQuery Sandbox:
# CREATE OR REPLACE TABLE, sin UPDATE.
# ============================================================

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

required_ind_839 = [
    "VP_ESCUELA",
    "VF_ESCUELA",
    "VS_ESCUELA",
]

missing_ind_839 = sorted(
    set(required_ind_839) - existing_columns
)

if missing_ind_839:
    raise RuntimeError(
        "No se puede crear INDICADOR_8_3_9. "
        "Faltan variables fuente: "
        + ", ".join(missing_ind_839)
    )

print("Variables fuente de INDICADOR_8_3_9 verificadas.")


select_prefix = (
    "* EXCEPT(INDICADOR_8_3_9)"
    if "INDICADOR_8_3_9" in existing_columns
    else "*"
)


indicador_839_sql = f"""
CREATE OR REPLACE TABLE `{A}` AS

SELECT
  {select_prefix},

  CASE
    WHEN VP_ESCUELA = 1
      OR VF_ESCUELA = 1
      OR VS_ESCUELA = 1
    THEN 1
    ELSE 0
  END AS INDICADOR_8_3_9

FROM `{A}`
"""

(SQL_DIR / "stage3_syntax_33_indicador_839.sql").write_text(
    indicador_839_sql,
    encoding="utf-8"
)

print(indicador_839_sql)

client.query(indicador_839_sql).result()

print("INDICADOR_8_3_9 creado correctamente.")


indicador_839_validation = client.query(f"""
SELECT
  COUNT(*) AS total_rows,

  COUNTIF(
    INDICADOR_8_3_9 IS NULL
    OR INDICADOR_8_3_9 NOT IN (0, 1)
  ) AS invalid_values,

  COUNTIF(INDICADOR_8_3_9 = 0) AS zero_values,
  COUNTIF(INDICADOR_8_3_9 = 1) AS one_values,

  COUNTIF(
    (VP_ESCUELA = 1 OR VF_ESCUELA = 1 OR VS_ESCUELA = 1)
    AND INDICADOR_8_3_9 != 1
  ) AS qualifying_case_not_one,

  COUNTIF(
    VP_ESCUELA != 1
    AND VF_ESCUELA != 1
    AND VS_ESCUELA != 1
    AND INDICADOR_8_3_9 != 0
  ) AS nonqualifying_case_not_zero

FROM `{A}`
""").result().to_dataframe()

indicador_839_validation.to_csv(
    LOG_DIR / "stage3_indicador_839_validation.csv",
    index=False
)

display(indicador_839_validation)

r = indicador_839_validation.iloc[0]

if r["total_rows"] != EXPECTED_ROWS:
    raise RuntimeError(
        "INDICADOR_8_3_9 alteró el universo analítico."
    )

if r["invalid_values"] > 0:
    raise RuntimeError(
        "INDICADOR_8_3_9 contiene valores fuera de 0/1."
    )

if (
    r["qualifying_case_not_one"] > 0
    or r["nonqualifying_case_not_zero"] > 0
):
    raise RuntimeError(
        "INDICADOR_8_3_9 no coincide con la lógica esperada."
    )

print(
    "INDICADOR_8_3_9 validado: "
    "18,807 filas, dominio 0/1 y consistencia exacta."
)


indicador_839_distribution = client.query(f"""
SELECT
  INDICADOR_8_3_9,
  COUNT(*) AS n
FROM `{A}`
GROUP BY INDICADOR_8_3_9
ORDER BY INDICADOR_8_3_9
""").result().to_dataframe()

indicador_839_distribution.to_csv(
    LOG_DIR / "stage3_indicador_839_distribution.csv",
    index=False
)

display(indicador_839_distribution)

Variables fuente de INDICADOR_8_3_9 verificadas.

CREATE OR REPLACE TABLE `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents` AS

SELECT
  *,

  CASE
    WHEN VP_ESCUELA = 1
      OR VF_ESCUELA = 1
      OR VS_ESCUELA = 1
    THEN 1
    ELSE 0
  END AS INDICADOR_8_3_9

FROM `enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents`

INDICADOR_8_3_9 creado correctamente.


,total_rows,invalid_values,zero_values,one_values,qualifying_case_not_one,nonqualifying_case_not_zero
0,18807,0,10210,8597,0,0


INDICADOR_8_3_9 validado: 18,807 filas, dominio 0/1 y consistencia exacta.


,INDICADOR_8_3_9,n
0,0,10210
1,1,8597


In [10]:
# ============================================================
# Validación final conjunta — SPSS 3.3
# Violencia en el entorno escolar
# ============================================================

indicators_33 = [
    "VP_ESCUELA",
    "VF_ESCUELA",
    "VP_o_VF_ESCUELA",
    "VP_VF_ESCUELA",
    "VS_ESCUELA",
    "INDICADOR_8_3_9",
]

# ------------------------------------------------------------
# 1. Verificar que existan todos los indicadores
# ------------------------------------------------------------

existing_columns = {
    field.name
    for field in client.get_table(A).schema
}

missing_33 = sorted(
    set(indicators_33) - existing_columns
)

if missing_33:
    raise RuntimeError(
        "No se puede cerrar el bloque 3.3. "
        "Faltan indicadores: "
        + ", ".join(missing_33)
    )

print("Todos los indicadores 3.3 existen.")


# ------------------------------------------------------------
# 2. Validar dominio y número de filas
# ------------------------------------------------------------

validation_parts = []

for variable in indicators_33:
    validation_sql = f"""
    SELECT
      '{variable}' AS variable,
      COUNT(*) AS total_rows,

      COUNTIF(
        `{variable}` IS NULL
        OR `{variable}` NOT IN (0, 1)
      ) AS invalid_values,

      COUNTIF(`{variable}` = 0) AS zero_values,
      COUNTIF(`{variable}` = 1) AS one_values

    FROM `{A}`
    """

    validation_parts.append(
        client.query(validation_sql)
        .result()
        .to_dataframe()
    )

validation_33 = pd.concat(
    validation_parts,
    ignore_index=True
)

validation_33.to_csv(
    LOG_DIR / "stage3_syntax_33_final_validation.csv",
    index=False
)

display(validation_33)

if (validation_33["total_rows"] != EXPECTED_ROWS).any():
    raise RuntimeError(
        "Algún indicador 3.3 alteró el universo analítico."
    )

if (validation_33["invalid_values"] > 0).any():
    raise RuntimeError(
        "Algún indicador 3.3 contiene valores fuera de 0/1."
    )


# ------------------------------------------------------------
# 3. Validar consistencia interna
# ------------------------------------------------------------

consistency_33 = client.query(f"""
SELECT
  COUNTIF(
    VP_o_VF_ESCUELA !=
    CASE
      WHEN VP_ESCUELA = 1
        OR VF_ESCUELA = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_vp_o_vf_escuela,

  COUNTIF(
    VP_VF_ESCUELA !=
    CASE
      WHEN VP_ESCUELA = 1
       AND VF_ESCUELA = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_vp_vf_escuela,

  COUNTIF(
    INDICADOR_8_3_9 !=
    CASE
      WHEN VP_ESCUELA = 1
        OR VF_ESCUELA = 1
        OR VS_ESCUELA = 1
      THEN 1
      ELSE 0
    END
  ) AS bad_indicador_839

FROM `{A}`
""").result().to_dataframe()

consistency_33.to_csv(
    LOG_DIR / "stage3_syntax_33_consistency_validation.csv",
    index=False
)

display(consistency_33)

if (consistency_33.iloc[0] > 0).any():
    raise RuntimeError(
        "Falló la consistencia interna del bloque 3.3."
    )


# ------------------------------------------------------------
# 4. Guardar resumen de cierre del notebook
# ------------------------------------------------------------

closure_33 = pd.DataFrame([
    {
        "block": "SPSS 3.3",
        "description": "Violencia en el entorno escolar",
        "table": A,
        "expected_rows": EXPECTED_ROWS,
        "validated_rows": int(
            validation_33["total_rows"].iloc[0]
        ),
        "indicators_created": len(indicators_33),
        "indicators": ";".join(indicators_33),
        "domain_check": "PASS",
        "consistency_check": "PASS",
        "run_utc": RUN_UTC,
    }
])

closure_33.to_csv(
    LOG_DIR / "stage3_syntax_33_closure.csv",
    index=False
)

display(closure_33)

print(
    "Bloque SPSS 3.3 completado: "
    "18,807 filas, seis indicadores y consistencia aprobada."
)

Todos los indicadores 3.3 existen.


,variable,total_rows,invalid_values,zero_values,one_values
0,VP_ESCUELA,18807,0,11285,7522
1,VF_ESCUELA,18807,0,16376,2431
2,VP_o_VF_ESCUELA,18807,0,10763,8044
3,VP_VF_ESCUELA,18807,0,16898,1909
4,VS_ESCUELA,18807,0,16657,2150
5,INDICADOR_8_3_9,18807,0,10210,8597


,bad_vp_o_vf_escuela,bad_vp_vf_escuela,bad_indicador_839
0,0,0,0


,block,description,table,expected_rows,validated_rows,indicators_created,indicators,domain_check,consistency_check,run_utc
0,SPSS 3.3,Violencia en el entorno escolar,enares-2024-crs04.enares2024_crs04_analytical....,18807,18807,6,VP_ESCUELA;VF_ESCUELA;VP_o_VF_ESCUELA;VP_VF_ES...,PASS,PASS,2026-07-17T02:00:03.319372+00:00


Bloque SPSS 3.3 completado: 18,807 filas, seis indicadores y consistencia aprobada.
